# 텍스트 분할 
- 언어 모델은 입력 길이에 제힌이 있다. 긴 문서를 적절하게 분할 하는 것이 중요하다.
- 텍스트 분할기를 사용하면 모델의 입력 제한을 준수할 수 있고 긴 문서로 인해 발생할 수 있는 처리 효율 저하를 방지한다. 

#### RecursiveCharacterTextSpliter 
- 긴 텍스트를 사용자가 지정한 최대 길이를 초과하지 않는 짧은 청크로 반복적으로 분할하는 도구이다. 
- 분할 작업은 내부적으로 지정된 구분자를 사용하며 이루어진다. 
- 기본 구분자는 ["\n\n", "\n", " ", "" ] 순서로 적용된다. 
- 가장 큰 구분자를 "\n\n"으로 나누구 분할된 청크가 여전히 길면 다음 구분자를 적용해서 점진적으로 작은 청크로 나누어 나간다. 

In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader 

loader = PyPDFLoader("./data/2024_KB_부동산_보고서_최종.pdf")
pages = loader.load()

print(f"총 글자수:  {len(''.join([i.page_content for i in pages]))}")


총 글자수:  82384


In [8]:
text_spliter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
texts =text_spliter.split_documents(pages)
print(f"분할된 청크 수 {len(texts)}")

texts[1].page_content
len(texts[1].page_content)

분할된 청크 수 218


468

#### SemanticChunker 
- SemanticChunker는 텍스트를 단순히 길이에 따라 나누는 것이 아닌 의미적으로 유사한 내용을 가진 청크로 분할하는 도구이다. 
- 텍스트를 문장 단위로 분할한 후 유사한 의미를 가진 문장들을 그룹화하여 하나의 청크로 구성한다. 

In [9]:
from langchain_experimental.text_splitter import SemanticChunker 
from langchain_openai.embeddings import OpenAIEmbeddings 
from langchain_community.document_loaders import PyPDFLoader

In [12]:
loader = PyPDFLoader('./data/2024_KB_부동산_보고서_최종.pdf')
pages = loader.load()

embeddings = OpenAIEmbeddings(
    model="bge-m3",
    base_url="http://localhost:1234/v1",
    api_key="lm-studio",
    check_embedding_ctx_length=False,
)

text_spliter = SemanticChunker(embeddings=embeddings)


chunks = text_spliter.split_documents(pages)
print(f"분할된 청크수 {len(chunks)}")

분할된 청크수 164


In [13]:
chunks[2].page_content


'매수 수요 위축으로 주택 매매 거래량이 급감하면서 향후 주택 경기에 대한 부정적 \n시각이 팽배하다. 무엇보다 여전히 높은 금리가 부담으로 작용하고 있다. 주택 경기 불황기에 고금리 \n부담은 주택 수요를 크게 위축시킬 수밖에 없기 때문이다. 다만 주택시장의 주요 변수들의 상황에 따라 \n소폭 반등 혹은 하락폭 확대 등의 방향이 정해질 것으로 보인다. 2024년 주택시장의 주요 변수는 공급과 금리다. 급격하게 위축된 주택 공급이 단기간에 증가하기는 \n쉽지 않으나 정부의 공급 시그널이 지속된다면 일정 부분 해소가 될 가능성이 있다. 무엇보다 금리가 주요 \n변수가 될 것이다. 기준금리 인하 시기와 인하 폭에 따라 주택 수요는 영향을 받을 수밖에 없기 때문이다. 한편, 수요 위축으로 거래가 급감한 상황에서 실수요자 금융 지원, 관련 규제 완화 등 수요 회복을 위한 \n정부 정책도 중요한 변수가 될 전망이다. \uf06e 7대 이슈를 통해 바라보는 2024년 주택시장 \n1 역대 최저 수준이 지속되고 있는 주택 거래 \n \n주택 매매 거래는 2022년에 이어 2년 연속 침체. 총선 이후 정책 불확실성 해소와 금리 인하로 인한 회\n복 가능성이 있으나 일부 지역 수요 쏠림 현상과 금리 인하 속도가 더딜 경우 회복세는 제한적일 전망 \n2 주택공급 급격한 감소로 인한 공급 부족 가능성 \n \n분양물량과 함께 장기적인 주택 공급 기반인 인허가물량까지 급감. 청약 수요 위축으로 분양 지연이 장기\n화될 가능성이 높은 가운데 정부의 공급 정책 구체화가 매우 중요 \n3 노후계획도시 특별법과 재건축 시장 영향 \n \n2023년 말 국회를 통과한 「노후계획도시 특별법」 시행을 앞두고 당초 51곳이었던 대상 지역이 108곳으\n로 확대. 단기적 효과는 제한적이나 사업진행이 구체화되면 시장 영향도 커질 것  \n4 전세 수요 아파트 집중, 입주물량 부족으로 가격 상승 가능성 확대 \n \n비아파트 전세 사기와 보증금 미반환 이슈 등으로 아파트로 전세 수요가 집중되는 가운데

In [15]:
chunks[3].page_content

'2 \n2024 KB 부동산 보고서: 2024년 주택시장 진단과 전망 \n \n \n \nExecutive Summary 2 \n\uf06e 주택 매매시장 하락 전망 우세, 부동산 투자 선호도 하락 \n• 2024년 주택 매매가격 지난해에 이어 올해도 하락 전망 우세, 높은 금리가 가장 큰 부담 \n부동산시장 전문가와 공인중개사, 자산관리전문가(PB)를 대상으로 한 설문 조사 결과, 2024년 전국 주택 \n매매가격은 하락세가 이어질 것이라는 전망이 우세하였다. 다만 시장 급락에 대한 우려는 다소 완화된 \n것으로 보인다. 상승에 대한 전망이 2023년 대비 크게 증가했기 때문이다(전문가 21%p, 공인중개사 \n17%p, PB 13%p  증가). 매매가격 하락요인으로는 높은 금리에 따른 이자부담이 가장 중요한 이유로 \n조사되었다. • 2024년 주택 전세가격, 비수도권은 하락 전망이 우세하나 수도권 전망은 엇갈려  \n2024년 전국 주택 전세가격에 대해 전문가의 53%, 공인중개사의 61%가 하락을 전망하였다. 하락폭에 \n대해서는 3% 이하가 될 것이라는 의견이 많았다. 다만 2023년 하락 전망이 압도적으로 우세했던 것과 \n달리 2024년에는 상승과 하락 전망에 격차가 크지 않았다. 지역별로 수도권에서는 주택 전세가격이 \n상승과 하락 의견이 엇갈렸으나 비수도권에서는 하락 전망이 우세하였다. • 경기 회복을 위해서는 금리 인하와 대출 규제 완화가 중요 변수 \n주택 경기 회복을 위해 필요한 핵심 정책으로 전문가와 공인중개사, PB 모두 금리 인하를 꼽았다. 다음으로 주택담보대출 지원, LTV·DSR 등 금융 규제 완화가 필요하다는 의견이 많았다. 특히 공인중개사 \n그룹에서 금리와 대출 관련 정책의 필요성을 높게 평가하였다.'

#### 의미 단위 분할 과정 
- 문장을 임베딩 벡터로 변환하여 각 문장의 의미를 숫자로 표현한다. 
- 인접한 문장 쌍 사이의 코사인 거리를 계산하여 문장 간의 의미적 차이가 어느 정도 인지 파악한다. 
- 분할 지정 결정 방식
    - 백분위수 방식 : 코사인 거리값이 설정한 백분위 수를 초과하는 지점을 분할지점으로 선택
    - 표준편차 방식 : 코사인 거리가 평균보다 특정 표준 편차 이상 크게 떨어진 분할 지점으로 선택 
    - 사분위수 방식 : 코사인 거리의 사분위 범위에 따라 지점을 결정한다. 